# 第 7 周 –「价格合适」顶点项目

本周目标：微调一个开源模型，根据产品描述估算价格。

## 课程日程（对照）

| 天 | 主题 |
|----|------|
| 第 1 天 | QLoRA 概念与动机 |
| 第 2 天 | 提示数据与基座模型 |
| 第 3 天 | 训练（上） |
| 第 4 天 | 训练（下） |
| 第 5 天 | 评估与对比 |

本笔记本侧重 **第 2 天（数据/提示）**，并给出第 3–5 天的 Colab 入口与结果图。


## 第 2 天：提示数据与基座模型

从 Hugging Face 加载商品数据集，配置分词器，统计 token 长度分布，再按截断阈值生成微调用的 **prompt / completion**。


In [ ]:
# ========== 定位 week7、导入定价工具与分词器 ==========
# 将 week7 添加到路径中，以便我们可以导入定价器
import sys
from pathlib import Path
# 从 cwd / 上一级 / 上两级寻找含 pricer 包的 week7 目录
for base in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
    week7_path = base / "week7"
    if (week7_path / "pricer").exists():
        # 找到后插入 sys.path 最前，优先 import
        sys.path.insert(0, str(week7_path))
        break

# 读环境变量（HF_TOKEN）
import os
# 加载 .env
from dotenv import load_dotenv
# Hugging Face 登录
from huggingface_hub import login
# 课程 Item：商品样本 + from_hub / make_prompts 等
from pricer.items import Item
# 笔记本友好进度条
from tqdm.notebook import tqdm
# 加载 Llama 分词器
from transformers import AutoTokenizer
# 画 token 直方图
import matplotlib.pyplot as plt


In [ ]:
# ========== 环境变量 + Hugging Face 登录 ==========
# False=全量数据集；True=lite 小集（更快）
LITE_MODE = False  # Set True for smaller dataset

# override=True：.env 覆盖已有环境变量
load_dotenv(override=True)
# 读取 HF_TOKEN（可为空）
hf_token = os.environ.get("HF_TOKEN")
if hf_token:
    # 有 token 则登录，便于拉门禁模型/数据集
    login(hf_token, add_to_git_credential=True)
else:
    # 提示手动 login；文案保持英文原样
    print("Warning: HF_TOKEN not set. Login manually with: login(token)")


In [ ]:
# ========== 从 Hub 拉取 train/val/test Item 列表 ==========
# 数据集作者命名空间
username = "ed-donner"
# lite / full 二选一
dataset = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"

# Item.from_hub：下载并反序列化为三个 Python 列表
train, val, test = Item.from_hub(dataset)
# 合并，方便后面统一统计 token
items = train + val + test

print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items")


In [ ]:
# ========== 加载与微调目标一致的基座分词器 ==========
# 基座模型 id：Llama 3.2 3B
BASE_MODEL = "meta-llama/Llama-3.2-3B"
# 只加载 tokenizer（本格不做模型权重）
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)


In [ ]:
# ========== 统计每条 summary 的 token 数 ==========
# count_tokens：用当前分词器量摘要长度；tqdm 显示进度
token_counts = [item.count_tokens(tokenizer) for item in tqdm(items)]


In [ ]:
# ========== 直方图：summary 的 token 长度分布 ==========
plt.figure(figsize=(15, 6))
# 标题里带上平均值与最大值（字符串格式保持原样）
plt.title(f"Tokens in Summary: Avg {sum(token_counts)/len(token_counts):,.1f} and highest {max(token_counts):,}\n")
plt.xlabel("Number of tokens in summary")
plt.ylabel("Count")
# 0–200、步长 10 的箱子
plt.hist(token_counts, rwidth=0.7, color="skyblue", bins=range(0, 200, 10))
plt.show()


In [ ]:
# ========== 选定截断阈值 CUTOFF，并估计会截断多少条 ==========
# 超过该 token 数的摘要将在 make_prompts 时被截断
CUTOFF = 110
# 统计超过阈值的条数
cut = len([count for count in token_counts if count > CUTOFF])
print(f"With CUTOFF={CUTOFF}, we will truncate {cut:,} items which is {cut/len(items):.1%}")


In [ ]:
# ========== 看一条训练样本的 summary 长什么样 ==========
print("Sample summary:")
print(train[0].summary)


In [ ]:
# ========== 为 train/val/test 生成 prompt（测试集不含答案泄露） ==========
# include_price=True：训练/验证提示里带价格答案，供 SFT
for item in tqdm(train + val):
    item.make_prompts(tokenizer, CUTOFF, True)
# 测试集 False：只给问题侧，评估时才比价格
for item in tqdm(test):
    item.make_prompts(tokenizer, CUTOFF, False)


In [ ]:
# ========== 抽查测试集第 0 条的 prompt / completion ==========
print("PROMPT:")
print(test[0].prompt)
print("\nCOMPLETION:")
print(test[0].completion)


In [ ]:
# ========== 统计「完整 prompt+completion」的 token 数 ==========
prompt_token_counts = [item.count_prompt_tokens(tokenizer) for item in tqdm(items)]


In [ ]:
# ========== 直方图：prompt+completion 总长度 ==========
plt.figure(figsize=(15, 6))
plt.title(f"Prompt+Completion Tokens: Avg {sum(prompt_token_counts)/len(prompt_token_counts):,.1f} and highest {max(prompt_token_counts):,}\n")
plt.xlabel("Number of tokens (prompt + completion)")
plt.ylabel("Count")
plt.hist(prompt_token_counts, rwidth=0.7, color="gold", bins=range(0, 200, 10))
plt.show()


In [ ]:
# ========== 可选：把生成好的 prompts 推到自己的 Hub（默认注释） ==========
# 可选：将提示推送到您的 HuggingFace 帐户以进行培训
# username = "your-hf-username"
# dataset = f"{username}/items_prompts_lite" if LITE_MODE else f"{username}/items_prompts_full"
# Item.push_prompts_to_hub(dataset, train, val, test)

# 或者使用 Ed 的公共数据集：
# https://huggingface.co/datasets/ed-donner/items_prompts_lite
# https://huggingface.co/datasets/ed-donner/items_prompts_full


## 第 3–4 天：用 QLoRA 训练

使用 **QLoRA（4bit 量化 + LoRA）** 微调 Llama 3.2 3B。**需要 GPU**（如 Google Colab T4/A100）。

推荐直接打开课程 Colab：

- [第 3–4 天训练 Colab](https://colab.research.google.com/drive/1fBTm_jzrFGr88PDOFTQF7JIlQ1JW15BG)
- [第 5 天评估 Colab](https://colab.research.google.com/drive/16e8aY_BlHjzzcR-2dCyDMCPdOQ8XeN1e)

若本机有 GPU，也可取消下一格注释，从 Hub 加载 prompts 后本地训练（下方仅示意骨架）。


In [ ]:
# ========== 训练骨架（默认注释）：QLoRA + SFTTrainer ==========
# 训练设置 - 在 Colab 或 GPU 机器上运行
# 取消注释并运行以加载提示数据集和训练

# from datasets import load_dataset
# from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
# from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
# from trl import SFTTrainer, SFTConfig
# import torch

# PROMPTS_DATASET = "ed-donner/items_prompts_lite"  # 或 items_prompts_full
# dataset = load_dataset(PROMPTS_DATASET)
# train_data = dataset["train"]
# val_data = dataset["val"]

# def format_example(ex): return {"text": ex["prompt"] + ex["completion"]}
# train_formatted = train_data.map(format_example)
# val_formatted = val_data.map(format_example)

# # 加载具有 4 位量化的基础模型
# bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16)
# model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=bnb_config, device_map="auto")
# tokenizer.pad_token = tokenizer.eos_token

# # LoRA 配置
# model = prepare_model_for_kbit_training(model)
# lora_config = LoraConfig(r=16, lora_alpha=32, target_modules=["q_proj", "v_proj"], lora_dropout=0.05, task_type="CAUSAL_LM")
# model = get_peft_model(model, lora_config)

# # SFTConfig + SFTTrainer - 然后 trainer.train()
# 本格默认可运行的只有下一行提示（完整流水线见上方 Colab）
print("See Colab links above for full training pipeline.")


## 第 5 天：评估

加载微调后的模型，在测试集上做价格预测，计算 **平均绝对误差（MAE）** 等指标。  
下一格演示如何接入课程的 `pricer.evaluator`（需本机已有微调权重与正确路径）。


In [ ]:
# ========== 尝试导入课程评估器；占位预测函数示例 ==========
# 使用 week7pricer.evaluator 进行评估（需要微调模型）
# 从 week7 目录运行或确保 week7 在路径上

try:
    from pricer.evaluator import evaluate, Tester
    def fine_tuned_pricer(item):
        # 替换为您的实际模型推理逻辑
        # 例如加载模型，分词器，在 item.test_prompt() 上运行生成
        return item.price  # placeholder
    # Tester 用函数名当图标题
    fine_tuned_pricer.__name__ = "fine_tuned_llama"
    # 评估（fine_tuned_pricer，测试，大小=200）
    # evaluate(fine_tuned_pricer, test, size=200)
    print("evaluate() and Tester available. Define your pricer and run evaluate(pricer, test).")
except ImportError as e:
    # 找不到包时给出路径提示（格式字符串保持原样）
    print(f"pricer.evaluator not found: {e}. Run from repo root or add week7 to path.")


## 结果：模型比较

下图汇总各类方法的预测误差（**平均绝对误差 MAE**）：常数基线、传统 ML、深度网络、前沿 LLM，以及本周微调的 Lite / Full 模型。


In [ ]:
# ========== Plotly 柱状图：各模型 MAE 一览 ==========
import plotly.graph_objects as go

# (显示名, 颜色, MAE)——数值来自作者实验，勿随意改以免与叙述对不上
results = [
    ("Constant", "gray", 106.18),
    ("Linear Regression", "gray", 101.56),
    ("NLP + LR", "gray", 76.81),
    ("Random Forest", "gray", 72.28),
    ("XGBoost", "gray", 68.23),
    ("Human (Ed)", "black", 87.62),
    ("Neural Network", "orange", 63.97),
    ("GPT 4.1 Nano", "slateblue", 62.51),
    ("Grok 4.1 Fast", "slateblue", 57.62),
    ("Gemini 3 Pro", "slateblue", 50.54),
    ("Claude 4.5 Sonnet", "slateblue", 47.10),
    ("GPT 5.1", "slateblue", 44.74),
    ("GPT 4.1 Nano (Fine-tuned)", "skyblue", 75.91),
    ("Deep Neural Network", "orange", 46.49),
    ("Base Llama 3.2 4 bit", "darkred", 110.72),
    ("Fine-tuned Lite", "red", 65.40),
    ("Fine-tuned Full", "red", 39.85)
]

# 拆成三条序列喂给 Plotly
labels, colors, values = zip(*results)

fig = go.Figure(go.Bar(x=labels, y=values, marker_color=colors))

fig.update_layout(
    title="Prediction error from each model",
    yaxis=dict(range=[0, max(values)], title="Error"),
    xaxis=dict(tickangle=-45),
    width=1000,
    height=800
)

fig.show()
